In [1]:
import matplotlib.pyplot as plt
import numpy as np
import os
import sys
import pickle

path = os.getcwd().split(os.sep + 'GUI')[0]
if path not in sys.path:
    print("not here")
    sys.path.append(path)

from neurolib.models.aln import ALNModel
from neurolib.utils import plotFunctions as plotFunc
from neurolib.utils import costFunctions as cost
import neurolib.dashboard.functions as functions
import neurolib.dashboard.data as data
    
# This will reload all imports as soon as the code changes
%load_ext autoreload
%autoreload 2 

#path = os.path.join(os.getcwd(), "plots")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [2]:
# read case
print(os.getcwd())
case = os.getcwd().split(os.sep)[-1]
print(case[0])

D:\Promotion\neurolib\GUI\current\gui\data\00011
0


### Bistability

In [3]:
aln = ALNModel()
N = aln.params.N

data.set_parameters(aln)

state_vars = aln.state_vars
init_vars = aln.init_vars

##############################################################
def setinit(init_vars_, model):
    state_vars = model.state_vars
    init_vars = model.init_vars
    for iv in range(len(init_vars)):
        for sv in range(len(state_vars)):
            if state_vars[sv] in init_vars[iv]:
                #print("set init vars ", )
                if model.params[init_vars[iv]].ndim == 2:
                    model.params[init_vars[iv]][0,:] = init_vars_[sv]
                else:
                    model.params[init_vars[iv]][0] = init_vars_[sv]
                    
##############################################################               
def setmaxmincontrol(max_c_c, min_c_c, max_c_r, min_c_r):
    import numpy as np
    
    max_cntrl = np.zeros(( 6 ))
    min_cntrl = np.zeros(( 6 ))
    
    max_cntrl[0] = max_c_c
    min_cntrl[0] = min_c_c
    max_cntrl[1] = max_c_c
    min_cntrl[1] = min_c_c
    max_cntrl[2] = max_c_r
    min_cntrl[2] = min_c_r
    max_cntrl[3] = max_c_r
    min_cntrl[3] = min_c_r
    max_cntrl[4] = max_c_r
    min_cntrl[4] = min_c_r
    max_cntrl[5] = max_c_r
    min_cntrl[5] = min_c_r
            
    return max_cntrl, min_cntrl

#####################################################
def getclosest(k_, found_solution, exc, inh, already_tried_):
    import numpy as np
    if len(found_solution) == 0:
        print("no solutions found")
        return -1
    
    start_ind = -1
    for j_ in found_solution:
        if j_ not in already_tried_ and j_ != k_:
            start_ind = j_
            break
            
    if start_ind == -1:
        return -1
        
    min_dist = np.sqrt((exc[k_] - exc[start_ind])**2 + (inh[k_] - inh[start_ind])**2)
    min_i = start_ind
        
    print(found_solution, already_tried_)
        
    if len(found_solution) == len(already_tried_):
        print("already tried all options")
        min_i = -1
        return min_i
    
    for i_ in found_solution:
        if i_ not in already_tried_:
            if i_ != k_ and i_ != min_i:
                dist_ = np.sqrt((exc[k_] - exc[i_])**2 + (inh[k_] - inh[i_])**2)
                if dist_ < min_dist:
                    min_dist = dist_
                    min_i = i_
                    
    if min_i == 0 and 0 in already_tried_:
        return -1
    
    return min_i

In [4]:
##### LOAD BOUNDARIES
data_file = 'bi.pickle'
with open(data_file,'rb') as f:
    load_array= pickle.load(f)
exc = load_array[0]
inh = load_array[1]
print(len(exc))
#plt.scatter(exc, inh)

147


In [5]:
bestControl_init = [None] * len(exc)
bestState_init = [None] * len(exc)
cost_init = [None] * len(exc)
runtime_init = [None] * len(exc)
grad_init = [None] * len(exc)
phi_init = [None] * len(exc)
costnode_init = [None] * len(exc)
weights_init = [None] * len(exc)

conv_init = [[False]*2] * len(exc)

In [6]:
bestControl_0 = [None] * len(exc)
bestState_0 = [None] * len(exc)
cost_0 = [None] * len(exc)
runtime_0 = [None] * len(exc)
grad_0 = [None] * len(exc)
phi_0 = [None] * len(exc)
costnode_0 = [None] * len(exc)
weights_0 = [None] * len(exc)

conv_0 = [[False]*2] * len(exc)

cgv_list = [None, "HS", "FR", "PR", "CD", "LS", "DY", "WYL", "HZ", None]

In [7]:
initVars = [None] * len(exc)
target = [None] * len(exc)
cost_uncontrolled = [None] * len(exc)

In [8]:
dur_pre = 10
dur_post = 10

n_pre = int(np.around(dur_pre/aln.params.dt + 1.,1))
n_post = int(np.around(dur_post/aln.params.dt + 1.,1))

tol = 1e-32
start_step = 10.
c_scheme = np.zeros(( 1,1 ))
c_scheme[0,0] = 1.
u_mat = np.identity(1)
u_scheme = np.array([[1.]])

c_var = [ [0], [1], [0,1]]
p_var = [ [0], [0], [0]]

### CURRENTS
cntrl_vars_0 = [0,1]
prec_vars = [0]

if case[0] == '0':    # low to high
    max_I = [3., -3.]
elif case[0] == '1':
    max_I = [-3., 3.]
    
if case[1] == '0':    # sparsity
    factor_ws = 1.
    factor_we = 0.
elif case[1] == '1':  # energy
    factor_ws = 0.
    factor_we = 1.
    
if case[3] == '0' and case[2] == '0':
    cntrl_vars_init = [0]
elif case[3] == '1' and case[2] == '0':
    cntrl_vars_init = [1]
elif case[3] == '0' and case[2] == '1':
    cntrl_vars_init = [2,4]
elif case[3] == '1' and case[2] == '1':
    cntrl_vars_init = [3,5]

if case[4] == '0':
    dur = 100
    trans_time = 0.8
elif case[4] == '1':
    dur = 400
    trans_time = 0.95
    
maxC = [5., -5., 0.18, 0.]

n_dur = int(np.around(dur/aln.params.dt + 1.,1))
max_cntrl, min_cntrl = setmaxmincontrol(maxC[0], maxC[1], maxC[2], maxC[3])

In [9]:
init_file = 'control_init_' + case + '.pickle'
final_file = 'control_0_' + case + '.pickle'
final_file_1 = 'control_1_' + case + '.pickle'

In [10]:
if os.path.isfile(init_file) :
    print("file found")
    
    with open(init_file,'rb') as f:
        load_array = pickle.load(f)

    bestControl_init = load_array[0]
    bestState_init = load_array[1]
    cost_init = load_array[2]
    runtime_init = load_array[3]
    grad_init = load_array[4]
    phi_init = load_array[5]
    costnode_init = load_array[6]
    weights_init = load_array[7]

file found


In [11]:
i_stepsize = 10
i_range = range(0, len(exc),i_stepsize)
data.set_parameters(aln)

for i in i_range:
    print("------- ", i, exc[i], inh[i])
    aln.params.mue_ext_mean = exc[i] * 5.
    aln.params.mui_ext_mean = inh[i] * 5.
    
    aln.params.duration = 3000.
    
    control0 = aln.getZeroControl()
    control0 = functions.step_control(aln, maxI_ = max_I[0])

    aln.run(control=control0)
    
    target_rates = np.zeros((2))
    target_rates[0] = aln.rates_exc[0,-1] 
    target_rates[1] = aln.rates_inh[0,-1]

    control0 = functions.step_control(aln, maxI_ = max_I[1])
    aln.run(control=control0)

    init_state_vars = np.zeros(( len(state_vars) ))
    for j in range(len(state_vars)):
        if aln.state[state_vars[j]].size == 1:
            init_state_vars[j] = aln.state[state_vars[j]][0]
        else:
            init_state_vars[j] = aln.state[state_vars[j]][0,-1]

    initVars[i] = init_state_vars
    
    aln.params.duration = dur

    target[i] = aln.getZeroTarget()
    target[i][:,0,:] = target_rates[0]
    target[i][:,1,:] = target_rates[1]

-------  0 0.4000000000000001 0.3500000000000001
-------  10 0.4250000000000001 0.42500000000000016
-------  20 0.4500000000000001 0.4750000000000002
-------  30 0.4250000000000001 0.5250000000000002
-------  40 0.5250000000000001 0.5500000000000003
-------  50 0.47500000000000014 0.6000000000000003
-------  60 0.5500000000000003 0.6250000000000003
-------  70 0.4500000000000001 0.6750000000000004
-------  80 0.5250000000000001 0.7000000000000004
-------  90 0.6000000000000003 0.7250000000000004
-------  100 0.4500000000000001 0.7750000000000005
-------  110 0.5000000000000002 0.8000000000000005
-------  120 0.5500000000000003 0.8250000000000005
-------  130 0.6000000000000003 0.8500000000000005
-------  140 0.4500000000000001 0.9000000000000006


In [12]:
data.set_parameters(aln)

for i in i_range:
    print("------- ", i, exc[i], inh[i])
    aln.params.mue_ext_mean = exc[i] * 5.
    aln.params.mui_ext_mean = inh[i] * 5.
    
    aln.params.duration = dur
        
    cost.setParams(1.0, 0.0, 0.0)

##### zero control as input for uncontrolled cost
    setinit(initVars[i], aln)
    control0 = aln.getZeroControl()

    # "HS", "FR", "PR", "HZ"
    cgv = None
    max_it = 0

    bestControl_init_, bestState_init_, cost_init_, runtime_init_, grad_init_, phi_init_, costnode_init_ = aln.A1(
        control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
        startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
        t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_init,
        prec_variables_ = prec_vars, transition_time_ = trans_time)
    
    cost_uncontrolled[i] = cost_init_[0]

-------  0 0.4000000000000001 0.3500000000000001
set cost params:  1.0 0.0 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  5902.406479238383
Gradient descend method:  None
RUN  0 , total integrated cost =  5902.406479238383
Improved over  0  iterations in  0.0  seconds by  0.0  percent.
-------  10 0.4250000000000001 0.42500000000000016
set cost params:  1.0 0.0 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  9111.456490210901
Gradient descend method:  None
RUN  0 , total integrated cost =  9111.456490210901
Improved over  0  iterations in  0.0  seconds by  0.0  percent.
-------  20 0.4500000000000001 0.4750000000000002
set cost params:  1.0 0.0 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  12738.116450271265
Gradient descend method:  None
RUN  0 , total integrated cost =  12738.116450271265
Improved over  0  iterations in  0.0  seconds by  0.0  percent.
-------  30 0.4250000000000001 0.52500000000000

In [13]:
i_range_ = []

for i in i_range:
    if type(bestControl_init[i]) == type(None):
        i_range_.append(i)

i_range = np.array(i_range_)
        
print(i_range)

[ 10  20  30  40  50  60  80  90 100 110 120 130]


In [14]:
factor_iteration = 20.

for i in i_range:
    print("------- ", i, exc[i], inh[i])
    aln.params.mue_ext_mean = exc[i] * 5.
    aln.params.mui_ext_mean = inh[i] * 5.
    
    aln.params.duration = dur
        
##### zero control as input for uncontrolled cost
    setinit(initVars[i], aln)
    control0 = aln.getZeroControl()

##### initial guess
    weight_ = 10
    cost.setParams(1.0, weight_ * factor_we, weight_ * factor_ws)

    setinit(initVars[i], aln)
    #if prev_i != -1:
    #    control0 = bestControl_init[prev_i][:,:,n_pre-1:-n_post+1]
    
    # "HS", "FR", "PR", "HZ"
    cgv = None
    max_it = int(100 * factor_iteration)
    
    weights_init[i] = cost.getParams()

    bestControl_init[i], bestState_init[i], cost_init[i], runtime_init[i], grad_init[i], phi_init[i], costnode_init[i] = aln.A1(
        control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
        startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
        t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_init,
        prec_variables_ = prec_vars, transition_time_ = trans_time)
    
    j = 1
    while cost_init[i][-j] == 0.:
        j += 1
    
    weight_ = 10 * cost_uncontrolled[i] / cost_init[i][-j]
    print("weight = ", weight_)
    cost.setParams(1.0, weight_ * factor_we, weight_ * factor_ws)

    setinit(initVars[i], aln)
    control0 = bestControl_init[i][:,:,n_pre-1:-n_post+1]

    # "HS", "FR", "PR", "HZ"
    cgv = None
    max_it = int(500 * factor_iteration)

    weights_init[i] = cost.getParams()
    
    bestControl_init[i], bestState_init[i], cost_init[i], runtime_init[i], grad_init[i], phi_init[i], costnode_init[i] = aln.A1(
        control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
        startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
        t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_init,
        prec_variables_ = prec_vars, transition_time_ = trans_time)
    
    prev_i = i

-------  10 0.4250000000000001 0.42500000000000016
set cost params:  1.0 0.0 10.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  9111.456490210901
Gradient descend method:  None
RUN  1 , total integrated cost =  777.7450427577537
RUN  2 , total integrated cost =  175.58970906556175
RUN  3 , total integrated cost =  136.12960985785867
RUN  4 , total integrated cost =  109.92686202528108
RUN  5 , total integrated cost =  93.03024402986588
RUN  6 , total integrated cost =  73.79540487389981
RUN  7 , total integrated cost =  59.838563463332584
RUN  8 , total integrated cost =  38.92873802793984
RUN  9 , total integrated cost =  30.01821002649804
RUN  10 , total integrated cost =  29.5266983282249
RUN  11 , total integrated cost =  29.27767521602492
RUN  12 , total integrated cost =  29.228544083797154
RUN  13 , total integrated cost =  29.222762239337506
RUN  14 , total integrated cost =  29.21286358389922
RUN  15 , total integrated cost =  29.20767921724479
RUN  1

ERROR:root:Problem in initial value trasfer


Control only changes marginally.
RUN  489 , total integrated cost =  24.817428550228108
Improved over  489  iterations in  94.5541729  seconds by  99.72762391416903  percent.
Problem in initial value trasfer:  Vmean_exc -56.64650685344838 -56.646506905294885
weight =  3671.3942670451056
set cost params:  1.0 0.0 3671.3942670451056
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  9091.970852227978
Gradient descend method:  None
RUN  1 , total integrated cost =  8971.174349478788
RUN  2 , total integrated cost =  8971.024725398938
RUN  3 , total integrated cost =  8970.634931695917
RUN  4 , total integrated cost =  8970.42385580711
RUN  5 , total integrated cost =  8960.778114878292
RUN  6 , total integrated cost =  8952.942134966626
RUN  7 , total integrated cost =  8952.75829760875
RUN  8 , total integrated cost =  8952.438280606619
RUN  9 , total integrated cost =  8952.403772338506
RUN  10 , total integrated cost =  8948.86746555517
RUN  11 , total integrated c

ERROR:root:Problem in initial value trasfer


Control only changes marginally.
RUN  98 , total integrated cost =  8942.474877080822
Improved over  98  iterations in  19.496168100000006  seconds by  1.6442636869048215  percent.
Problem in initial value trasfer:  Vmean_exc -56.64592496519824 -56.64593480682769
-------  20 0.4500000000000001 0.4750000000000002
set cost params:  1.0 0.0 10.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  12738.116450271265
Gradient descend method:  None
RUN  1 , total integrated cost =  1800.0786222783458
RUN  2 , total integrated cost =  745.0360522735323


...

RUN  11 , total integrated cost =  39270.30012722848
Control only changes marginally.
RUN  11 , total integrated cost =  39270.30012722848
Improved over  11  iterations in  2.6967884999999683  seconds by  0.15530922558710358  percent.
Problem in initial value trasfer:  Vmean_exc -56.699650126169566 -56.69965011528709
-------  100 0.4500000000000001 0.7750000000000005
set cost params:  1.0 0.0 10.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  10559.709248318897
Gradient descend method:  None
RUN  1 , total integrated cost =  10559.709248318897
Control only changes marginally.
RUN  1 , total integrated cost =  10559.709248318897
Improved over  1  iterations in  0.3722725000000082  seconds by  0.0  percent.
weight =  10.0
set cost params:  1.0 0.0 10.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  10559.709248318897
Gradient descend method:  None
RUN  1 , total integrated cost =  10559.709248318897
Control only changes marginally.
RUN  

ERROR:root:Problem in initial value trasfer


RUN  6 , total integrated cost =  289.0554048219748
Control only changes marginally.
RUN  6 , total integrated cost =  289.0554048219748
Improved over  6  iterations in  1.6960162999999966  seconds by  99.25361441732949  percent.
Problem in initial value trasfer:  Vmean_exc -56.69978363158897 -56.69980877082395
weight =  1339.790080647147
set cost params:  1.0 0.0 1339.790080647147
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  29579.292072648077
Gradient descend method:  None


ERROR:root:Problem in initial value trasfer


RUN  1 , total integrated cost =  29579.292072648077
Control only changes marginally.
RUN  1 , total integrated cost =  29579.292072648077
Improved over  1  iterations in  0.508861300000035  seconds by  0.0  percent.
Problem in initial value trasfer:  Vmean_exc -56.69978363158897 -56.69980877082395


In [15]:
#plot initial guesses
"""
for i in i_range:
        
    aln.params.mue_ext_mean = exc[i] * 5.
    aln.params.mui_ext_mean = inh[i] * 5.

    plotFunc.plot_control_current(aln, [bestControl_init[i]],
        [costnode_init[i]], [weights_init[i]], dur,
        dur_pre, dur_post, initVars[i], target[i], '', filename_ = '', transition_time_ = trans_time,
        labels_ = ["init", "sparse control" + str(i)], print_cost_ = False)
"""

'\nfor i in i_range:\n        \n    aln.params.mue_ext_mean = exc[i] * 5.\n    aln.params.mui_ext_mean = inh[i] * 5.\n\n    plotFunc.plot_control_current(aln, [bestControl_init[i]],\n        [costnode_init[i]], [weights_init[i]], dur,\n        dur_pre, dur_post, initVars[i], target[i], \'\', filename_ = \'\', transition_time_ = trans_time,\n        labels_ = ["init", "sparse control" + str(i)], print_cost_ = False)\n'

In [16]:
found_solution = []
no_solution = []
last_update = -1
factor_iteration = 20.
already_tried = [ [] for _ in range(len(exc)) ]
i_range = range(0, len(exc),i_stepsize)
i_range_0 = []
i_range_1 = []

print(already_tried, len(already_tried))

for k in range(len(i_range)**2):
    print('------------------------------------------------------------')
    print('--------------------', k)
    print('------------------------------------------------------------')
        
    print("found solution: ", found_solution)
    print("no solution: ", no_solution)
    
    if len(i_range) == len(found_solution) + len(no_solution):
        print("found solution for all parameters")
        break
    
    #if last_update != k-1:
    #    print("no improvement from previous step")
    #    break

    for i in i_range:
        print("------- ", i, exc[i], inh[i])

        if np.abs(bestState_init[i][0,0,-1] - target[i][0,0,-1]) < 0.5 * np.abs(
            bestState_init[i][0,0,-1] - bestState_init[i][0,0,0]):
            if i not in found_solution:
                print("found solution for ", i)
                found_solution.append(i)
                #last_update = k
            if i in no_solution:
                no_solution.pop(no_solution.index(i))
            continue

        #if i not in no_solution:
        #    print("no solution for ", i)
        #    no_solution.append(i)
        
        if i not in i_range_0:
            i_range_0.append(i)
            
        if i not in i_range_1:
            i_range_1.append(i)

        closest_ = getclosest(i, found_solution, exc, inh, already_tried[i])
        print("closest index ", closest_)

        weight_ = 10
        cost.setParams(1.0, weight_ * factor_we, weight_ * factor_ws)

        setinit(initVars[i], aln)
        aln.params.mue_ext_mean = exc[i] * 5.
        aln.params.mui_ext_mean = inh[i] * 5.
            
        if i != 0 and closest_ != -1:
            control0 = bestControl_init[closest_][:,:,n_pre-1:-n_post+1]
            if closest_ not in already_tried[i]:
                already_tried[i].append(closest_)
                        
        if closest_ == -1:
            print("all options tried already")
            if i not in no_solution:
                no_solution.append(i)
                continue

        # "HS", "FR", "PR", "HZ"
        cgv = None
        max_it = int(100 * factor_iteration)

        weights_init[i] = cost.getParams()
        
        print("precision vars = ", prec_vars)

        bestControl_init[i], bestState_init[i], cost_init[i], runtime_init[i], grad_init[i], phi_init[i], costnode_init[i] = aln.A1(
            control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
            startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
            t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_init,
            prec_variables_ = prec_vars, transition_time_ = trans_time)

        j = 1
        while cost_init[i][-j] == 0.:
            j += 1

        weight_ = 10 * cost_uncontrolled[i] / cost_init[i][-j]
        print("weight = ", weight_)
        cost.setParams(1.0, weight_ * factor_we, weight_ * factor_ws)

        setinit(initVars[i], aln)
        control0 = bestControl_init[i][:,:,n_pre-1:-n_post+1]

        # "HS", "FR", "PR", "HZ"
        cgv = None
        max_it = int(500 * factor_iteration)

        weights_init[i] = cost.getParams()

        bestControl_init[i], bestState_init[i], cost_init[i], runtime_init[i], grad_init[i], phi_init[i], costnode_init[i] = aln.A1(
            control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
            startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
            t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_init,
            prec_variables_ = prec_vars, transition_time_ = trans_time)

[[], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], []] 147
------------------------------------------------------------
-------------------- 0
------------------------------------------------------------
found solution:  []
no solution:  []
-------  0 0.4000000000000001 0.3500000000000001
found solution for  0
-------  10 0.4250000000000001 0.42500000000000016
found solution for  10
-------  20 0.4500000000000001 0.4750000000000002
found solution for  20
-------

ERROR:root:Problem in initial value trasfer


Control only changes marginally.
RUN  62 , total integrated cost =  741.5147815770124
Improved over  62  iterations in  30.648504300000013  seconds by  0.7176833789019383  percent.
Problem in initial value trasfer:  Vmean_exc -56.635347539141726 -56.63539139242091
weight =  107.59485016357786
set cost params:  1.0 0.0 107.59485016357786
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  4672.176800642374
Gradient descend method:  None
RUN  1 , total integrated cost =  4669.2216587740295
RUN  2 , total integrated cost =  4669.183678953787


ERROR:root:Problem in initial value trasfer


RUN  3 , total integrated cost =  4669.183678953787
Control only changes marginally.
RUN  3 , total integrated cost =  4669.183678953787
Improved over  3  iterations in  3.7885698999999704  seconds by  0.06406268033724416  percent.
Problem in initial value trasfer:  Vmean_exc -56.63535429521428 -56.635398062095966
-------  40 0.5250000000000001 0.5500000000000003
found solution for  40
-------  50 0.47500000000000014 0.6000000000000003
[0, 10, 20, 40] []
closest index  40
set cost params:  1.0 0.0 10.0
precision vars =  [0]
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  15975.93056981847
Gradient descend method:  None
RUN  1 , total integrated cost =  15944.348728626115
RUN  2 , total integrated cost =  73.6194136779184
RUN  3 , total integrated cost =  65.79763668155753
RUN  4 , total integrated cost =  65.41755804750022
RUN  5 , total integrated cost =  65.29118606072687
RUN  6 , total integrated cost =  65.21020283087603
RUN  7 , total integrated cost =  65.

...

Control only changes marginally.
RUN  93 , total integrated cost =  15822.150855622207
Improved over  93  iterations in  18.613874299999907  seconds by  0.6527934225476599  percent.
Problem in initial value trasfer:  Vmean_exc -56.683223063750255 -56.68322465309795
-------  60 0.5500000000000003 0.6250000000000003
found solution for  60
-------  70 0.4500000000000001 0.6750000000000004
found solution for  70
-------  80 0.5250000000000001 0.7000000000000004
[0, 10, 20, 40, 60, 70] []
closest index  70
set cost params:  1.0 0.0 10.0
precision vars =  [0]
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  91.80650678241815
Gradient descend method:  None
RUN  1 , total integrated cost =  74.61899626394985
RUN  2 , total integrated cost =  73.78891786449792
RUN  3 , total integrated cost =  73.7736590698286
RUN  4 , total integrated cost =  73.77294130969442
RUN  5 , total integrated cost =  73.77291090337368
RUN  6 , total integrated cost =  73.77291046149797
RUN  7 ,

RUN  6 , total integrated cost =  118.23710336294661
RUN  7 , total integrated cost =  118.23710253887866
RUN  8 , total integrated cost =  118.2371004892343
RUN  9 , total integrated cost =  118.23709943869441
RUN  10 , total integrated cost =  118.2370845918002
RUN  11 , total integrated cost =  118.23706761676931
RUN  12 , total integrated cost =  118.23706663378648
RUN  13 , total integrated cost =  118.23706468352833
RUN  14 , total integrated cost =  118.23706370993028
RUN  15 , total integrated cost =  118.2370460954604
RUN  16 , total integrated cost =  118.23702569423824
RUN  17 , total integrated cost =  118.23697686181046
RUN  18 , total integrated cost =  118.23693898056518
RUN  19 , total integrated cost =  118.23693810907034
RUN  20 , total integrated cost =  118.23693616227274
RUN  30 , total integrated cost =  118.23684364514546
RUN  40 , total integrated cost =  118.2364175521676
RUN  50 , total integrated cost =  118.2361654986464
RUN  60 , total integrated cost =  11

ERROR:root:Problem in initial value trasfer


Control only changes marginally.
RUN  7722 , total integrated cost =  10552.284112963416
Improved over  7722  iterations in  1454.4209920999997  seconds by  0.07026639451127892  percent.
Problem in initial value trasfer:  Vmean_exc -56.65536819487426 -56.65536830239059
-------  110 0.5000000000000002 0.8000000000000005
[0, 10, 20, 40, 60, 70, 90] []
closest index  90
set cost params:  1.0 0.0 10.0
precision vars =  [0]
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  19248.196694532697
Gradient descend method:  None
RUN  1 , total integrated cost =  19226.32169941821
RUN  2 , total integrated cost =  19226.1019729132
RUN  3 , total integrated cost =  19226.098478186002
RUN  4 , total integrated cost =  19226.098323213897
RUN  5 , total integrated cost =  19226.09831837102
RUN  6 , total integrated cost =  19226.09831820667
RUN  7 , total integrated cost =  19226.098318201577
RUN  8 , total integrated cost =  19226.098318201537
RUN  9 , total integrated cost =  19

RUN  700 , total integrated cost =  103.97483177865705
RUN  800 , total integrated cost =  103.97473508162558
RUN  900 , total integrated cost =  103.9746393717598
RUN  1000 , total integrated cost =  103.97458379388144
RUN  1100 , total integrated cost =  103.97445558568297
RUN  1200 , total integrated cost =  103.97386190387341
RUN  1300 , total integrated cost =  103.97375373885518
RUN  1400 , total integrated cost =  103.97368160328759
RUN  1500 , total integrated cost =  103.97359226511418
RUN  1600 , total integrated cost =  103.8561169483165
RUN  1700 , total integrated cost =  103.85605994108153
RUN  1800 , total integrated cost =  103.85598712551054
RUN  1900 , total integrated cost =  103.85591096730924
RUN  2000 , total integrated cost =  103.85583533835621
RUN  2000 , total integrated cost =  103.85583533835621
Improved over  2000  iterations in  155.18608159999985  seconds by  11.904682617114261  percent.
weight =  1851.2294716579029
set cost params:  1.0 0.0 1851.22947165

In [ ]:
factor_iteration = 20
full_converge = False
conv_init = [[False]*2] * len(exc)

for i in range(len(conv_init)):
    if i not in i_range:
        conv_init[i] = [True, True]
        
counter = 0

while full_converge == False:
    
    if counter > 100:
        break
        
    print(conv_init[::i_stepsize])
    full_converge = True
    
    for conv in conv_init[::i_stepsize]:
        if not conv[0]:
            full_converge = False
            break
        if not conv[1]:
            full_converge = False
            break
    
    if full_converge:
        print("full convergence")
        break

    for i in i_range:        

        print("------- ", i, exc[i], inh[i])
        aln.params.mue_ext_mean = exc[i] * 5.
        aln.params.mui_ext_mean = inh[i] * 5.
        
        j = 1
        while cost_init[i][-j] == 0.:
            j += 1
            
        if j == cost_init[i].shape[0]-1:
            print("converged for ", i)
            if conv_init[i][0]:
                conv_init[i] = [True, True]
            else:
                conv_init[i] = [True, False]
            continue
    
        print("no convergence")
            
        weight_ = (factor_we * weights_init[i][1] * cost_uncontrolled[i] / cost_init[i][-j]
                   + factor_ws * weights_init[i][2] * cost_uncontrolled[i] / cost_init[i][-j]) - 1
        print("weight = ", weight_)
        cost.setParams(1.0, weight_ * factor_we, weight_ * factor_ws)

        setinit(initVars[i], aln)
        control0 = bestControl_init[i][:,:,n_pre-1:-n_post+1]

        # "HS", "FR", "PR", "HZ"
        cgv = None
        max_it = int( 500 * factor_iteration )

        weights_init[i] = cost.getParams()

        bestControl_init[i], bestState_init[i], cost_init[i], runtime_init[i], grad_init[i], phi_init[i], costnode_init[i] = aln.A1(
            control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
            startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
            t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_init,
            prec_variables_ = prec_vars, transition_time_ = trans_time)
        
        counter += 1

[[False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False]]
-------  0 0.4000000000000001 0.3500000000000001
converged for  0
-------  10 0.4250000000000001 0.42500000000000016
no convergence
weight =  3739.770824900679
set cost params:  1.0 0.0 3739.770824900679
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  9108.596956107462
Gradient descend method:  None
RUN  1 , total integrated cost =  9108.596026881347
RUN  2 , total integrated cost =  9108.595894124113
RUN  3 , total integrated cost =  9108.595866469315
RUN  4 , total integrated cost =  9108.595858164314
RUN  5 , total integrated cost =  9108.595855166986
RUN  6 , total integrated cost =  9108.59585390852
RUN  7 , total integrated cost =  9108.595853155295
RUN  8 , total integrated cost =  9108.595852690925
RUN  9 , total inte

ERROR:root:Problem in initial value trasfer


Control only changes marginally.
RUN  44 , total integrated cost =  9108.595851799419
Improved over  44  iterations in  3.8175191999998788  seconds by  1.2123799621122089e-05  percent.
Problem in initial value trasfer:  Vmean_exc -56.64591947733186 -56.64592941055265
-------  20 0.4500000000000001 0.4750000000000002
converged for  20
-------  30 0.4250000000000001 0.5250000000000002
no convergence
weight =  182.84923377528693
set cost params:  1.0 0.0 182.84923377528693
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  7695.940219930212
Gradient descend method:  None
RUN  1 , total integrated cost =  7694.043138241359
RUN  2 , total integrated cost =  7694.043138241358


ERROR:root:Problem in initial value trasfer


RUN  3 , total integrated cost =  7694.043138241358
Control only changes marginally.
RUN  3 , total integrated cost =  7694.043138241358
Improved over  3  iterations in  1.6949952999993911  seconds by  0.02465042132138251  percent.
Problem in initial value trasfer:  Vmean_exc -56.635372757262694 -56.635416285652546
-------  40 0.5250000000000001 0.5500000000000003
no convergence
weight =  7726.342857680706
set cost params:  1.0 0.0 7726.342857680706
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  25527.465063550568
Gradient descend method:  None
RUN  1 , total integrated cost =  25527.463723559256
RUN  2 , total integrated cost =  25527.46359493181
RUN  3 , total integrated cost =  25527.46350575406
RUN  4 , total integrated cost =  25527.463484822558
RUN  5 , total integrated cost =  25527.463484025764
RUN  6 , total integrated cost =  25527.463484025753


...

RUN  2 , total integrated cost =  24409.821971934725
RUN  3 , total integrated cost =  24409.82195839126
RUN  4 , total integrated cost =  24409.821955856634
RUN  5 , total integrated cost =  24409.821950969585
RUN  6 , total integrated cost =  24409.82193742611
RUN  7 , total integrated cost =  24409.821934891494
RUN  8 , total integrated cost =  24409.82193000447
RUN  9 , total integrated cost =  24409.821916460965
RUN  10 , total integrated cost =  24409.821913926353
RUN  11 , total integrated cost =  24409.821909039325
RUN  12 , total integrated cost =  24409.821895495814
RUN  13 , total integrated cost =  24409.8218929612
RUN  14 , total integrated cost =  24409.821888074213
RUN  15 , total integrated cost =  24409.821874530684
RUN  16 , total integrated cost =  24409.821871996086
RUN  17 , total integrated cost =  24409.8218671091
RUN  18 , total integrated cost =  24409.821853565594
RUN  19 , total integrated cost =  24409.821851031018
RUN  20 , total integrated cost =  24409.82

RUN  6000 , total integrated cost =  24409.78017230287
RUN  7000 , total integrated cost =  24409.77320794966
RUN  8000 , total integrated cost =  24409.76653586491
RUN  9000 , total integrated cost =  24409.76030718109
RUN  10000 , total integrated cost =  24409.754079109465
RUN  10000 , total integrated cost =  24409.754079109465
Improved over  10000  iterations in  796.9343704999992  seconds by  0.0002782502626672567  percent.
-------  90 0.6000000000000003 0.7250000000000004
converged for  90
-------  100 0.4500000000000001 0.7750000000000005
converged for  100
-------  110 0.5000000000000002 0.8000000000000005
converged for  110
-------  120 0.5500000000000003 0.8250000000000005
converged for  120
-------  130 0.6000000000000003 0.8500000000000005
converged for  130
-------  140 0.4500000000000001 0.9000000000000006
converged for  140
[[True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [False, False], [True, True], [True

RUN  70 , total integrated cost =  24409.821605107216
RUN  80 , total integrated cost =  24409.8215456752
RUN  90 , total integrated cost =  24409.821477194666
RUN  100 , total integrated cost =  24409.821418226868
RUN  110 , total integrated cost =  24409.821358680587
RUN  120 , total integrated cost =  24409.821289938434
RUN  130 , total integrated cost =  24409.821231420603
RUN  140 , total integrated cost =  24409.821172449327
RUN  150 , total integrated cost =  24409.82110427219
RUN  160 , total integrated cost =  24409.821045233974
RUN  170 , total integrated cost =  24409.82098580316
RUN  180 , total integrated cost =  24409.820917323497
RUN  190 , total integrated cost =  24409.820858356576
RUN  200 , total integrated cost =  24409.820798811546
RUN  300 , total integrated cost =  24409.820170210674
RUN  400 , total integrated cost =  24409.819551844386
RUN  500 , total integrated cost =  24409.818933035076
RUN  600 , total integrated cost =  24409.818305028588
RUN  700 , total 

RUN  1 , total integrated cost =  24409.822062847736
RUN  2 , total integrated cost =  24409.822058570608
RUN  3 , total integrated cost =  24409.822056671983
RUN  4 , total integrated cost =  24409.82204208186
RUN  5 , total integrated cost =  24409.822036155438
RUN  6 , total integrated cost =  24409.8220340809
RUN  7 , total integrated cost =  24409.82202107592
RUN  8 , total integrated cost =  24409.822017001396
RUN  9 , total integrated cost =  24409.82201503567
RUN  10 , total integrated cost =  24409.822000217147
RUN  11 , total integrated cost =  24409.821994593403
RUN  12 , total integrated cost =  24409.82199311091
RUN  13 , total integrated cost =  24409.821967941018
RUN  14 , total integrated cost =  24409.821952968497
RUN  15 , total integrated cost =  24409.821951593374
RUN  16 , total integrated cost =  24409.821916476885
RUN  17 , total integrated cost =  24409.82189127914
RUN  18 , total integrated cost =  24409.82188993569
RUN  19 , total integrated cost =  24409.8218

RUN  5000 , total integrated cost =  24409.770930146933
RUN  6000 , total integrated cost =  24409.761636976866
RUN  7000 , total integrated cost =  24409.752671191127
RUN  8000 , total integrated cost =  24409.744393379864
RUN  9000 , total integrated cost =  24409.735714336173
RUN  10000 , total integrated cost =  24409.72718531517
RUN  10000 , total integrated cost =  24409.72718531517
Improved over  10000  iterations in  764.720218699993  seconds by  0.0003888817515047549  percent.
-------  90 0.6000000000000003 0.7250000000000004
converged for  90
-------  100 0.4500000000000001 0.7750000000000005
converged for  100
-------  110 0.5000000000000002 0.8000000000000005
converged for  110
-------  120 0.5500000000000003 0.8250000000000005
converged for  120
-------  130 0.6000000000000003 0.8500000000000005
converged for  130
-------  140 0.4500000000000001 0.9000000000000006
converged for  140
[[True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True,

In [ ]:
print(conv_init[::i_stepsize])

with open(init_file,'wb') as f:
    pickle.dump([bestControl_init, bestState_init, cost_init, runtime_init, grad_init, phi_init,
                 costnode_init, weights_init], f)

In [ ]:
if os.path.isfile(final_file) :
    print("file found")
    
    with open(final_file,'rb') as f:
        load_array = pickle.load(f)

    bestControl_0 = load_array[0]
    bestState_0 = load_array[1]
    cost_0 = load_array[2]
    runtime_0 = load_array[3]
    grad_0 = load_array[4]
    phi_0 = load_array[5]
    costnode_0 = load_array[6]
    weights_0 = load_array[7]

In [ ]:
i_range_0 = range(0, len(exc),i_stepsize)
i_range_ = []

for i in i_range_0:
    if type(bestControl_0[i]) == type(None):
        i_range_.append(i)

i_range_0 = np.array(i_range_)
        
print(i_range_0)

In [ ]:
factor_iteration = 20
    
for i in i_range_0:
    print("------- ", i, exc[i], inh[i])
    aln.params.mue_ext_mean = exc[i] * 5.
    aln.params.mui_ext_mean = inh[i] * 5.

# exc and inh control current 

    setinit(initVars[i], aln)
    aln.params.duration = dur
    control0 = bestControl_init[i][:,:,n_pre-1:-n_post+1].copy()
    #control0 = bestControl_0[i][:,:,n_pre-1:-n_post+1]
 
    cost.setParams(1.0, 0.0, 10.0)

    # "HS", "FR", "PR", "HZ"
    cgv = None
    max_it = 50 * factor_iteration

    weights_0[i] = cost.getParams()

    bestControl_0[i], bestState_0[i], cost_0[i], runtime_0[i], grad_0[i], phi_0[i], costnode_0[i] = aln.A1(
        control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
        startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
        t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_0,
        prec_variables_ = prec_vars, transition_time_ = trans_time)
    
    j = 1
    while cost_0[i][-j] == 0.:
        j += 1
    
    weight_ = 100
    print("weight = ", weight_)
    cost.setParams(1.0, weight_ * factor_we, weight_ * factor_ws)

    setinit(initVars[i], aln)
    control0 = bestControl_0[i][:,:,n_pre-1:-n_post+1]

    # "HS", "FR", "PR", "HZ"
    cgv = "HS"
    max_it = 50 * factor_iteration

    weights_0[i] = cost.getParams()

    bestControl_0[i], bestState_0[i], cost_0[i], runtime_0[i], grad_0[i], phi_0[i], costnode_0[i] = aln.A1(
        control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
        startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
        t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_0,
        prec_variables_ = prec_vars, transition_time_ = trans_time)
    
    j = 1
    while cost_0[i][-j] == 0.:
        j += 1
    
    weight_ = 100 * cost_uncontrolled[i] / cost_0[i][-j] - 1
    print("weight = ", weight_)
    cost.setParams(1.0, weight_ * factor_we, weight_ * factor_ws)

    setinit(initVars[i], aln)
    control0 = bestControl_0[i][:,:,n_pre-1:-n_post+1]

    # "HS", "FR", "PR", "HZ"
    cgv = None
    max_it = 50 * factor_iteration

    weights_0[i] = cost.getParams()

    bestControl_0[i], bestState_0[i], cost_0[i], runtime_0[i], grad_0[i], phi_0[i], costnode_0[i] = aln.A1(
        control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
        startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
        t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_0,
        prec_variables_ = prec_vars, transition_time_ = trans_time)

In [ ]:
"""
for i in i_range_0:
    
    aln.params.mue_ext_mean = exc[i] * 5.
    aln.params.mui_ext_mean = inh[i] * 5.

    plotFunc.plot_control_current(aln, [bestControl_init[i], bestControl_0[i]],
        [costnode_init[i], costnode_0[i]], [weights_init[i], weights_0[i]], dur,
        dur_pre, dur_post, initVars[i], target[i], '', filename_ = '', transition_time_ = trans_time,
        labels_ = ["init", "sparse control" + str(i)], print_cost_ = False)
"""

In [ ]:
factor_iteration = 20
full_converge = False
conv_0 = [[False]*2] * len(exc)

for i in range(len(conv_0)):
    if i not in i_range_0:
        conv_0[i] = [True, True]

counter = 0

while full_converge == False:
    print('---------------', counter)
    
    if counter > 100:
        break
    
    print(conv_0[::i_stepsize])
    full_converge = True
    
    for conv in conv_0[::i_stepsize]:
        if not conv[0]:
            full_converge = False
            break
        if not conv[1]:
            full_converge = False
            break
    
    if full_converge:
        print("full convergence")
        break

    for i in i_range_0:

        print("------- ", i, exc[i], inh[i])
        aln.params.mue_ext_mean = exc[i] * 5.
        aln.params.mui_ext_mean = inh[i] * 5.

        j = 1
        while cost_0[i][-j] == 0.:
            j += 1
            
        if j == cost_0[i].shape[0]-1:
            print("converged for ", i)
            if conv_0[i][0]:
                conv_0[i] = [True, True]
            else:
                conv_0[i] = [True, False]
            continue
    
        print("no convergence")

        weight_ = (factor_we * weights_0[i][1] * cost_uncontrolled[i] / cost_0[i][-j]
                       + factor_ws * weights_0[i][2] * cost_uncontrolled[i] / cost_0[i][-j]) - 1
        print("weight = ", weight_)
        cost.setParams(1.0, weight_ * factor_we, weight_ * factor_ws)

        setinit(initVars[i], aln)
        control0 = bestControl_0[i][:,:,n_pre-1:-n_post+1]

        # "HS", "FR", "PR", "HZ"
        cgv = None
        max_it = 500 * factor_iteration

        weights_0[i] = cost.getParams()

        bestControl_0[i], bestState_0[i], cost_0[i], runtime_0[i], grad_0[i], phi_0[i], costnode_0[i] = aln.A1(
            control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
            startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
            t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_0,
            prec_variables_ = prec_vars, transition_time_ = trans_time)
        
    counter += 1
    

In [ ]:
print(conv_0[::i_stepsize])

with open(final_file,'wb') as f:
    pickle.dump([bestControl_0, bestState_0, cost_0, runtime_0, grad_0, phi_0,
                 costnode_0, weights_0], f)

In [ ]:
bestControl_1 = [None] * len(exc)
bestState_1 = [None] * len(exc)
cost_1 = [None] * len(exc)
runtime_1 = [None] * len(exc)
grad_1 = [None] * len(exc)
phi_1 = [None] * len(exc)
costnode_1 = [None] * len(exc)
weights_1 = [None] * len(exc)

conv_1 = [[False]*2] * len(exc)

In [ ]:
if os.path.isfile(final_file_1) :
    print("file found")
    
    with open(final_file_1,'rb') as f:
        load_array = pickle.load(f)

    bestControl_1 = load_array[0]
    bestState_1 = load_array[1]
    cost_1 = load_array[2]
    runtime_1 = load_array[3]
    grad_1 = load_array[4]
    phi_1 = load_array[5]
    costnode_1 = load_array[6]
    weights_1 = load_array[7]

In [ ]:
i_range_1 = range(0, len(exc),i_stepsize)
i_range_ = []

for i in i_range_1:
    if type(bestControl_1[i]) == type(None):
        i_range_.append(i)

i_range_1 = np.array(i_range_)  

print(i_range_1)

In [ ]:
factor_iteration = 20

for i in i_range_1:        

    print("------- ", i, exc[i], inh[i])
    aln.params.mue_ext_mean = exc[i] * 5.
    aln.params.mui_ext_mean = inh[i] * 5.

    cost.setParams(1.0, 1. * factor_we, 1. * factor_ws)

    setinit(initVars[i], aln)
    control0 = bestControl_0[i][:,:,n_pre-1:-n_post+1]

    # "HS", "FR", "PR", "HZ"
    cgv = None
    max_it = int( 500 * factor_iteration )

    weights_1[i] = cost.getParams()

    bestControl_1[i], bestState_1[i], cost_1[i], runtime_1[i], grad_1[i], phi_1[i], costnode_1[i] = aln.A1(
        control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
        startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
        t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_init,
        prec_variables_ = prec_vars, transition_time_ = trans_time)

In [ ]:
factor_iteration = 20
full_converge = False

for i in range(len(conv_1)):
    if i not in i_range_1:
        conv_1[i] = [True, True]
        
counter = 0

while full_converge == False:
    
    print('---------------', counter)
    
    print(conv_1[::i_stepsize])
    full_converge = True
    
    for conv in conv_1[::i_stepsize]:
        if not conv[0]:
            full_converge = False
            break
        if not conv[1]:
            full_converge = False
            break
    
    if full_converge:
        print("full convergence")
        break

    for i in i_range_1:        

        print("------- ", i, exc[i], inh[i])
        aln.params.mue_ext_mean = exc[i] * 5.
        aln.params.mui_ext_mean = inh[i] * 5.
        
        j = 1
        while cost_1[i][-j] == 0.:
            j += 1
            
        if j == cost_1[i].shape[0]-1:
            print("converged for ", i)
            if conv_1[i][0]:
                conv_1[i] = [True, True]
            else:
                conv_1[i] = [True, False]
            continue
    
        print("no convergence")
        
        cost.setParams(1.0, 1. * factor_we, 1. * factor_ws)

        setinit(initVars[i], aln)
        control0 = bestControl_1[i][:,:,n_pre-1:-n_post+1]

        # "HS", "FR", "PR", "HZ"
        cgv = None
        max_it = int( 500 * factor_iteration )

        weights_1[i] = cost.getParams()

        bestControl_1[i], bestState_1[i], cost_1[i], runtime_1[i], grad_1[i], phi_1[i], costnode_1[i] = aln.A1(
            control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
            startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
            t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_init,
            prec_variables_ = prec_vars, transition_time_ = trans_time)
        
        counter += 1

In [ ]:
print(conv_1[::i_stepsize])

with open(final_file_1,'wb') as f:
    pickle.dump([bestControl_1, bestState_1, cost_1, runtime_1, grad_1, phi_1,
                 costnode_1, weights_1], f)